# AI Research Agent — Colab project

In this notebook you build a research pipeline with three stages: a
**researcher** that gathers facts with web tools, an **analyst** that
interprets the findings, and a **writer** that produces the final report.
You build the agents yourself with LangChain's `create_agent`, and you
chain them into a workflow yourself with LangGraph's Graph API
(`StateGraph`). The tools, the tracing helpers, and the shared runner are
given to you — you only write the parts marked TODO. There are three
TODOs: write the prompts (TODO #1), build the three agents (TODO #2),
and build the pipeline (TODO #3).

**Google Colab (easiest):** open this notebook in Colab, add a secret
named `OPENROUTER_API_KEY` (the key icon in the left sidebar, or
`Runtime → Edit session secrets`), then run the cells top to bottom
(`Runtime → Run all`) after you finish the TODOs.

**On your own machine:** run `uv sync` in this folder, then
`uv run jupyter lab research_agent.ipynb`, and put your key in a `.env`
file (copy `.env.example`).

In [1]:
# On your own machine (not Colab) run: uv sync — and skip this cell
!pip install -q "langchain>=1.4" "langgraph>=1.2.11" "langchain-openai>=1.2" beautifulsoup4 requests python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.8/161.8 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 17.2 MB/s eta 0:00:00


## 1. Setup

This cell is complete — just run it. It reads your API key (from Colab
Secrets, from `.env`, or by asking you) and creates `llm`, the model
object every agent in this notebook will use.

In [2]:
import getpass
import os
from typing import TypedDict

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import ToolCallLimitMiddleware
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph

try:
    # Running in Google Colab? Read the key from Colab Secrets.
    from google.colab import userdata
    try:
        # userdata.get raises SecretNotFoundError if the secret is not set;
        # fall through to the getpass prompt below instead of erroring.
        api_key = userdata.get('OPENROUTER_API_KEY') or ''
    except Exception:
        api_key = ''
except ImportError:
    # Running locally? Read the key from a .env file.
    load_dotenv()
    api_key = os.environ.get('OPENROUTER_API_KEY', '')

if not api_key:
    # Last resort: ask for the key in this cell.
    api_key = getpass.getpass('Paste your OPENROUTER_API_KEY: ')
if not api_key:
    raise ValueError('Set OPENROUTER_API_KEY (Colab Secrets, .env, or paste it when asked)')

MODEL_NAME = 'deepseek/deepseek-v4-flash'
llm = ChatOpenAI(model=MODEL_NAME, base_url='https://openrouter.ai/api/v1', api_key=api_key)

## 2. Observability helpers (given, run as-is)

Every agent run below prints a trace tree: one line per run, with the
token count and the real cost on it. Later you can swap this stub for real
Langfuse by changing one import — the function names match Langfuse's API.

In [3]:
# A small stand-in for Langfuse's @observe decorator.
# Swap to real Langfuse later by changing only the import, e.g.:
#     from langfuse import observe
import asyncio
import functools
import time
import uuid
from contextvars import ContextVar
from dataclasses import dataclass, field
from typing import Any, List, Optional


@dataclass
class Span:
    id: str
    name: str
    start_time: float
    level: int
    type: str = 'span'
    parent: Optional['Span'] = None
    children: List['Span'] = field(default_factory=list)
    end_time: Optional[float] = None
    input: Any = None
    output: Any = None
    metadata: dict = field(default_factory=dict)
    usage: dict = field(default_factory=dict)
    model: Optional[str] = None


_current_span: ContextVar[Optional[Span]] = ContextVar('current_span', default=None)


def print_tree(span: 'Span'):
    """Print one span and all of its children, indented by depth."""
    duration = (span.end_time - span.start_time) * 1000
    indent = '  ' * span.level
    prefix, suffix = ('=== TRACE: ', ' ===') if span.level == 0 else ('|-- ', '')
    meta_parts = []
    if span.model:
        meta_parts.append(f'model={span.model}')
    if span.usage.get('total_tokens'):
        meta_parts.append(f"tokens={span.usage['total_tokens']}")
    if 'cost_usd' in span.metadata:
        meta_parts.append(f"${span.metadata['cost_usd']:.4f}")
    meta_str = f" [{', '.join(meta_parts)}]" if meta_parts else ''
    type_str = f" [{span.type}]" if span.type != 'span' else ''
    print(f'{indent}{prefix}{span.name}{type_str}{suffix} ({duration:.2f}ms){meta_str}')
    for child in span.children:
        print_tree(child)


def _make_span(span_name: str, span_type: str, args, kwargs) -> 'Span':
    parent = _current_span.get()
    level = parent.level + 1 if parent else 0
    span = Span(id=str(uuid.uuid4())[:8], name=span_name, type=span_type,
                start_time=time.time(), level=level, parent=parent,
                input={'args': args, 'kwargs': kwargs})
    if parent:
        parent.children.append(span)
    return span


def _finish_span(span: 'Span'):
    span.end_time = time.time()
    if span.level == 0:
        print('\n' + '-' * 60)
        print_tree(span)
        print('-' * 60 + '\n')


def observe(name=None, as_type=None):
    """Decorator that records a span around a sync or async function."""
    def decorator(func):
        span_name = name if isinstance(name, str) else func.__name__
        span_type = as_type or 'span'

        if asyncio.iscoroutinefunction(func):
            @functools.wraps(func)
            async def wrapper(*args, **kwargs):
                span = _make_span(span_name, span_type, args, kwargs)
                token = _current_span.set(span)
                try:
                    result = await func(*args, **kwargs)
                    if span.output is None:
                        span.output = result
                    return result
                except Exception as e:
                    span.output = f'Error: {e}'
                    raise
                finally:
                    _finish_span(span)
                    _current_span.reset(token)
        else:
            @functools.wraps(func)
            def wrapper(*args, **kwargs):
                span = _make_span(span_name, span_type, args, kwargs)
                token = _current_span.set(span)
                try:
                    result = func(*args, **kwargs)
                    if span.output is None:
                        span.output = result
                    return result
                except Exception as e:
                    span.output = f'Error: {e}'
                    raise
                finally:
                    _finish_span(span)
                    _current_span.reset(token)

        return wrapper

    # Support both @observe and @observe(name='foo', as_type='bar')
    if callable(name):
        func, name = name, None
        return decorator(func)
    return decorator


class LangfuseContext:
    """Attach usage/model/cost data to the span currently being recorded."""

    def update_current_observation(self, **kwargs):
        span = _current_span.get()
        if not span:
            return
        if isinstance(kwargs.get('usage'), dict):
            span.usage.update(kwargs['usage'])
        if 'model' in kwargs:
            span.model = kwargs['model']
        if isinstance(kwargs.get('metadata'), dict):
            span.metadata.update(kwargs['metadata'])


langfuse_context = LangfuseContext()

In [4]:
# LoopDetector: spots when an agent repeats itself.
# check_tool_call flags repeated tool calls; check_output_stagnation flags
# stage outputs that keep coming back nearly identical.
from dataclasses import dataclass


@dataclass
class LoopDetectionResult:
    is_looping: bool
    strategy: str  # 'exact', 'fuzzy', 'stagnation', or 'none'
    message: str
    confidence: float


class LoopDetector:
    """Detects agent loops using exact match, fuzzy match, and stagnation."""

    def __init__(self, exact_threshold: int = 2, fuzzy_threshold: float = 0.8,
                 stagnation_window: int = 3):
        self.exact_threshold = exact_threshold
        self.fuzzy_threshold = fuzzy_threshold
        self.stagnation_window = stagnation_window
        self.tool_history: list[tuple[str, str]] = []  # (tool_name, args_str)
        self.output_history: list[str] = []

    def _jaccard_similarity(self, s1: str, s2: str) -> float:
        """Word-level overlap between two strings: shared words / all words."""
        tokens1 = set(s1.lower().split())
        tokens2 = set(s2.lower().split())
        if not tokens1 and not tokens2:
            return 1.0
        if not tokens1 or not tokens2:
            return 0.0
        return len(tokens1 & tokens2) / len(tokens1 | tokens2)

    def check_tool_call(self, tool_name: str, tool_input: str) -> LoopDetectionResult:
        """Call this BEFORE running a tool. Flags identical or near-identical repeats."""
        current = (tool_name, tool_input.strip())
        exact_count = sum(1 for past_tool, past_input in self.tool_history
                          if (past_tool, past_input.strip()) == current)
        if exact_count >= self.exact_threshold:
            self.tool_history.append(current)
            return LoopDetectionResult(
                is_looping=True, strategy='exact', confidence=1.0,
                message=(f"Exact loop detected: '{tool_name}' called {exact_count + 1} "
                         f'times with identical arguments. Change your approach.'))

        recent_history = self.tool_history[-5:]
        fuzzy_matches = sum(1 for past_tool, past_input in recent_history
                            if past_tool == tool_name
                            and self._jaccard_similarity(tool_input, past_input) >= self.fuzzy_threshold)
        if fuzzy_matches >= self.exact_threshold:
            self.tool_history.append(current)
            return LoopDetectionResult(
                is_looping=True, strategy='fuzzy', confidence=0.85,
                message=(f"Fuzzy loop detected: '{tool_name}' called with very similar "
                         f'arguments {fuzzy_matches + 1} times. Try a different approach.'))

        self.tool_history.append(current)
        return LoopDetectionResult(is_looping=False, strategy='none', message='', confidence=0.0)

    def check_output_stagnation(self, output: str) -> LoopDetectionResult:
        """Flags when the last few outputs are all nearly the same text."""
        self.output_history.append(output)
        if len(self.output_history) < self.stagnation_window:
            return LoopDetectionResult(is_looping=False, strategy='none', message='', confidence=0.0)

        recent = self.output_history[-self.stagnation_window:]
        similarities = [self._jaccard_similarity(recent[i], recent[j])
                        for i in range(len(recent)) for j in range(i + 1, len(recent))]
        avg_similarity = sum(similarities) / len(similarities) if similarities else 0
        if avg_similarity >= self.fuzzy_threshold:
            return LoopDetectionResult(
                is_looping=True, strategy='stagnation', confidence=avg_similarity,
                message=(f'Output stagnation detected: last {self.stagnation_window} '
                         f'outputs are {avg_similarity:.0%} similar. The agent is not '
                         f'making progress. Try a different approach entirely.'))
        return LoopDetectionResult(is_looping=False, strategy='none', message='', confidence=0.0)

    def reset(self):
        self.tool_history.clear()
        self.output_history.clear()

## 3. Research tools (given)

The `@tool` decorator turns a plain Python function into a tool the agent
can call. The function's name, docstring, and type hints become the schema
the model sees, and the returned string becomes the tool's result. This
cell is complete — run it.

In [5]:
import logging
import socket
from urllib.parse import urlparse

import requests
from bs4 import BeautifulSoup

logger = logging.getLogger(__name__)


def validate_url(url: str) -> bool:
    """
    Validate URL to prevent SSRF (Server-Side Request Forgery).
    Blocks localhost, private IP ranges, and non-http/https schemes.
    """
    try:
        parsed = urlparse(url)
        if parsed.scheme not in ["http", "https"]:
            return False

        hostname = parsed.hostname
        if not hostname:
            return False

        # Resolve hostname to IP
        try:
            ip_address = socket.gethostbyname(hostname)
        except socket.gaierror:
            return False # Could not resolve

        # Simple check for private ranges (10.x.x.x, 192.168.x.x, 172.16.x.x, 127.x.x.x)
        # In a real prod env, use the `ipaddress` module for strict checking
        parts = ip_address.split('.')
        if parts[0] == '10': return False
        if parts[0] == '192' and parts[1] == '168': return False
        if parts[0] == '172' and 16 <= int(parts[1]) <= 31: return False
        if parts[0] == '127': return False
        if ip_address == "0.0.0.0": return False

        return True
    except Exception:
        return False


@tool("search_web")
def search_web(query: str, max_results: int = 5) -> str:
    """Search the web for a query. Returns a list of results with title, link, and snippet."""
    url = "https://html.duckduckgo.com/html/"
    headers = {"User-Agent": "Mozilla/5.0"}

    try:
        response = requests.post(url, data={"q": query}, headers=headers, timeout=10)
        response.raise_for_status()
    except Exception as e:
        logger.error(f"Search request failed: {e}")
        # Return an error string gracefully instead of crashing the agent
        return f"Search failed: {e}"

    logger.info(f"Searching web for: '{query}'")

    soup = BeautifulSoup(response.text, "html.parser")
    lines = []
    for result in soup.find_all("div", class_="result", limit=max_results):
        title_tag = result.find("a", class_="result__a")
        snippet_tag = result.find("a", class_="result__snippet")

        if title_tag and snippet_tag:
            link = title_tag["href"]
            # Basic validation on the result link too
            if validate_url(link):
                lines.append(
                    f"{len(lines) + 1}. {title_tag.get_text(strip=True)}\n"
                    f"   Link: {link}\n"
                    f"   Snippet: {snippet_tag.get_text(strip=True)}"
                )

    logger.info(f"Search returned {len(lines)} results for '{query}'")
    if not lines:
        logger.warning(f"No results found for '{query}' (Raw response length: {len(response.text)})")
        return f"No results found for '{query}'."

    return "\n\n".join(lines)


@tool("read_webpage")
def read_webpage(url: str) -> str:
    """Read the content of a webpage. Returns the text content."""
    if not validate_url(url):
        return "Error: Invalid or restricted URL. Access to local/private networks is blocked."

    try:
        if "example.com" in url:
             return f"Simulated content for {url}."

        logger.info(f"Reading webpage: {url}")
        headers = {"User-Agent": "Mozilla/5.0"}
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")
        # Remove script and style elements
        for script in soup(["script", "style"]):
            script.decompose()

        text = soup.get_text(separator="\n")
        # Clean up whitespace
        lines = (line.strip() for line in text.splitlines())
        chunks = (phrase.strip() for line in lines for phrase in line.split("  "))
        text = '\n'.join(chunk for chunk in chunks if chunk)

        content = text[:10000]  # Truncate to avoid context overflow
        logger.info(f"Read {len(content)} chars from {url}")
        return content

    except Exception as e:
        return f"Error reading {url}: {e}"


# The tool list the researcher agent gets.
RESEARCH_TOOLS = [search_web, read_webpage]

In [6]:
# Quick smoke test that the tool works. Needs internet.
# If your network blocks DuckDuckGo, this prints an error string — that is fine.
print(search_web.invoke({'query': 'what is langgraph'}))

1. What is LangGraph - GeeksforGeeks
   Link: https://www.geeksforgeeks.org/machine-learning/what-is-langgraph/
   Snippet: LangGraphisan open-source framework from LangChain designed to build and manage AI agent workflows using graph-based structures. It allows developers to define workflows as nodes and edges, making complex agent interactions more structured, scalable and easier to control.

2. LangGraph: Agent Orchestration Framework for Reliable AI Agents - LangChain
   Link: https://www.langchain.com/langgraph
   Snippet: Control agent workflows withLangGraph: durable execution, memory, streaming, and human-in-the-loop.

3. What is LangGraph? - IBM
   Link: https://www.ibm.com/think/topics/langgraph
   Snippet: LangGraph, created by LangChain, is an open source AI agent framework designed to build, deploy and manage complex generative AI agent workflows. It provides a set of tools and libraries that enable users to create, run and optimize large language models (LLMs) in a scalab

## 4. The shared runner (given)

`run_agent` is the one function that calls any agent you build. It invokes
a compiled agent, pulls the final answer off the message list, adds up the
token usage, and records usage and the real cost from OpenRouter on the trace. It returns:

```python
{'answer': <final text>, 'metadata': {'total_messages': n, 'total_tokens': n}}
```

You will call `run_agent` from the pipeline nodes you write in TODO #3.

In [7]:
@observe(name='agent_run', as_type='agent')
async def run_agent(agent, query: str, max_steps: int = 10) -> dict:
    result = await agent.ainvoke(
        {'messages': [('user', query)]},
        config={'recursion_limit': 2 * max_steps + 1},
    )
    messages = result['messages']
    answer = messages[-1].content
    total_tokens = sum((m.usage_metadata or {}).get('total_tokens', 0) for m in messages if getattr(m, 'usage_metadata', None))
    langfuse_context.update_current_observation(usage={'total_tokens': total_tokens}, model=MODEL_NAME)
    # Real cost in USD: OpenRouter returns it in each response's usage object.
    total_cost = sum((m.response_metadata or {}).get('token_usage', {}).get('cost') or 0.0 for m in messages)
    langfuse_context.update_current_observation(metadata={'cost_usd': round(total_cost, 6)})
    return {'answer': answer, 'metadata': {'total_messages': len(messages), 'total_tokens': total_tokens}}

## 5. TODO #1 — prompts

Each agent's behavior comes from its system prompt:

- **Researcher**: gathers raw facts. It should call the web tools, collect
  relevant information, and report its findings with the links it used.
- **Analyst**: reads the research notes. It should find patterns, compare
  the points, and write a short structured analysis. It has no tools.
- **Writer**: reads the analysis. It should turn it into a clear final
  report for a beginner reader, with headings and a conclusion.

In [18]:
# Tell it to use the search_web and read_webpage tools and report findings with links.
RESEARCHER_PROMPT = """You are the research agent in a multi-agent research pipeline.

Your job is to gather enough reliable information to answer the research question and then STOP.

Use the available research tools carefully.

Research workflow:
1. Start with search_web to find relevant sources.
2. Select only the most relevant sources.
3. Use read_webpage to inspect the actual content of those sources.
4. After you have enough information from a few good sources, STOP using tools and write the research notes.

IMPORTANT TOOL RULES:
- Do not call the same tool with the same arguments more than once.
- Never read the same webpage more than once.
- Do not repeatedly search for the same question.
- Use no more than 2 search_web calls.
- Use no more than 3 read_webpage calls.
- You do not need to read every search result.
- If one webpage cannot be read, move to another source instead of retrying it repeatedly.
- Once you have useful information from 2–3 sources, stop researching and produce the notes.

SOURCE RULES:
- For numerical claims, rankings, dates, investments, targets, or major announcements:
  - Prefer primary or official sources when available.
  - Do not present a number as confirmed unless the source actually supports it.
  - Keep track of which source supports each important claim.
  - If a claim comes from only one secondary source, make that clear.

Your final response must contain research notes, not a conversation with the user.

For each important finding:
- State the finding clearly.
- Identify the source.
- Include the source link when available.

If sources disagree, clearly describe the disagreement instead of choosing a side without evidence.

Do not say that you need research notes from the user.
Do not ask the user for documents.
Do not discuss your internal tool process.

The final response should contain substantive research findings whenever the tools returned useful information.
"""

# Tell it to read the research notes and write a short structured analysis (no tools).
ANALYST_PROMPT = """You are the analysis agent in a research pipeline.

The researcher output will be provided to you as research notes along with the original question.

Treat the provided research notes as the actual research material you must analyze. Do not ask the user to provide research notes or documents.

Turn the research notes into a short and clear analysis.

Compare the information from different sources. Look for:
- common points
- disagreements
- missing information
- unsupported or unreliable claims

Do not invent information that is not present in the research notes.

Organize the analysis into:
1. Key findings
2. Agreements and disagreements
3. Missing information or limitations

Only use NEEDS_MORE_RESEARCH when important information is genuinely missing and additional web research could reasonably find it.

If more research is needed, start your response with:

NEEDS_MORE_RESEARCH:

Then explain exactly what information is missing.

Otherwise, provide the analysis directly."""

# Tell it to turn the analysis into a clear final report for a beginner reader.
WRITER_PROMPT = """You are the writer agent in a research pipeline.

Your job is to turn the provided research and analysis into a clear final report for someone who is new to the topic.

Use only the research notes and analysis provided to you.
Do not invent facts, numbers, sources, or conclusions that are not supported by the provided material.

Write the report in a clear and easy-to-understand way.

Requirements:
- Use clear headings to organize the report.
- Explain difficult terms when they first appear.
- Present the main findings clearly.
- Distinguish documented facts from interpretations, projections, or claims made by sources.
- Do not turn an ambition, target, announcement, or projection into a confirmed result.
- If sources disagree or the research identifies limitations, mention them clearly.
- Do not make broad evaluative claims unless they are directly supported by the provided research.
- Keep the report concise but informative.
- Include important source links when they are available in the research notes.
- Do not ask the user for research notes or additional information.
- Do not mention internal agents, prompts, tools, or the research pipeline.

End the report with a short conclusion that summarizes the main takeaway in one or two sentences.
"""

## 6. TODO #2 — build the three agents

`create_agent` gives you the complete agent loop: a model node, a tools
node, and the routing between them. You do NOT write the loop yourself —
the model decides when to call a tool and when to answer.
`ToolCallLimitMiddleware` is the step budget: it stops the agent after a
fixed number of tool calls.

Here are the exact shapes to build (the analyst and writer have no tools,
so they need no middleware):

```python
researcher = create_agent(
    model=llm,
    tools=RESEARCH_TOOLS,
    system_prompt=RESEARCHER_PROMPT,
    middleware=[ToolCallLimitMiddleware(thread_limit=10, exit_behavior='end')],
)
analyst = create_agent(model=llm, tools=[], system_prompt=ANALYST_PROMPT)
writer = create_agent(model=llm, tools=[], system_prompt=WRITER_PROMPT)
```

In [19]:
# TODO #2: replace each None with your create_agent call (pattern above).
from contextvars import ContextVar
from langchain.agents.middleware import wrap_tool_call
from langchain_core.messages import ToolMessage
tool_loop_detector: ContextVar[LoopDetector | None] = ContextVar(
    "tool_loop_detector",
    default=None,
)

@wrap_tool_call
async def detect_repeated_tool_calls(request, handler):
    detector = tool_loop_detector.get()

    if detector is not None:
        tool_name = request.tool_call["name"]
        tool_input = str(request.tool_call.get("args", {}))

        check = detector.check_tool_call(tool_name, tool_input)

        if check.is_looping:
            print("Loop warning:", check.message)

    return await handler(request)

researcher = create_agent(
    model=llm,
    tools=RESEARCH_TOOLS,
    system_prompt=RESEARCHER_PROMPT,
    middleware=[
        ToolCallLimitMiddleware(
            thread_limit=10,
            exit_behavior="end"
        ),
        detect_repeated_tool_calls,
    ],
)


analyst = create_agent(model=llm, tools=[], system_prompt=ANALYST_PROMPT)
writer = create_agent(model=llm, tools=[], system_prompt=WRITER_PROMPT)

## 7. TODO #3 — build the pipeline with the Graph API

`create_agent` builds ONE agent. To chain several agents into a workflow,
use LangGraph's Graph API: `StateGraph`. You define a shared state (a
`TypedDict`), one node function per stage, and the edges that say what
runs after what. The pipeline is: researcher → analyst → writer.

Here is the recipe:

```python
class PipelineState(TypedDict):
    query: str    # the original question
    report: str   # the text passed from stage to stage

async def researcher_node(state: PipelineState) -> dict:
    result = await run_agent(researcher, state['query'], 10)
    return {'report': result['answer']}

async def analyst_node(state: PipelineState) -> dict:
    result = await run_agent(analyst, state['report'], 10)
    return {'report': result['answer']}

async def writer_node(state: PipelineState) -> dict:
    result = await run_agent(writer, state['report'], 10)
    return {'report': result['answer']}

graph = StateGraph(PipelineState)
graph.add_node('researcher', researcher_node)
graph.add_node('analyst', analyst_node)
graph.add_node('writer', writer_node)
graph.add_edge(START, 'researcher')
graph.add_edge('researcher', 'analyst')
graph.add_edge('analyst', 'writer')
graph.add_edge('writer', END)
pipeline = graph.compile()

result = await pipeline.ainvoke({'query': query, 'report': ''})
```

Each node returns a dict of state updates; LangGraph merges them into the
shared state before the next node runs.

**The sequential pipeline above is a starting point — it cannot pass on
its own.** A straight line caps your architecture grade below the
passing threshold (see EVALUATION.md). To pass, change the graph: run
stages in parallel, send the writer's draft back to the researcher when
the fact-checker is not satisfied, or add a planner node that splits the
query first.

In [20]:
from langgraph.graph import START, END, StateGraph
from langgraph.checkpoint.memory import InMemorySaver

async def run_pipeline(query: str) -> dict:
    assert researcher is not None and analyst is not None and writer is not None, 'finish TODO #2 first'
    class PipelineState(TypedDict):
        query: str
        research: str
        analysis: str
        report: str
        retry_count: int
        stage_answers: list[str]

    MAX_RETRIES = 2
    MAX_AGENT_STEPS = 10

    stage_detector = LoopDetector()

    def check_stage_output(answer: str) -> None:
        check = stage_detector.check_output_stagnation(answer)

        if check.is_looping:
            print("Loop warning:", check.message)

    async def researcher_node(state: PipelineState) -> dict:
        if state["retry_count"] > 0:
            missing = state["analysis"].split(
                "NEEDS_MORE_RESEARCH:", 1
            )[-1].strip()

            task = f"""
Original research question:
{state["query"]}

The previous analysis identified missing information:

{missing}

Conduct additional web research specifically to fill these gaps.

Return clear research notes containing:
- important findings
- relevant source information
- source links

If sources disagree, clearly mention the disagreement.
"""

        else:
            task = f"""
Research this question:

{state["query"]}

Find relevant and recent sources using the available research tools.
Read the actual webpages rather than relying only on search snippets.

Return clear research notes containing:
- important findings
- relevant source information
- source links

If sources disagree, clearly mention the disagreement.
"""

        attempt_detector = LoopDetector()
        token = tool_loop_detector.set(attempt_detector)

        try:
            result = await run_agent(
                researcher,
                task,
                MAX_AGENT_STEPS,
            )
        finally:
            tool_loop_detector.reset(token)

        answer = result["answer"]

        check = attempt_detector.check_output_stagnation(answer)

        if check.is_looping:
            print("Loop warning:", check.message)

        return {
            "research": answer,
            "stage_answers": state["stage_answers"] + [answer],
        }

    async def analyst_node(state: PipelineState) -> dict:
        task = f"""
You are analyzing research that was produced by another agent.

Original research question:
{state["query"]}

================ RESEARCH NOTES ================
{state["research"]}
================ END RESEARCH NOTES ================

Treat the text between RESEARCH NOTES and END RESEARCH NOTES
as the actual research material you must analyze.

Do NOT ask the user to provide research notes.
Do NOT say that research notes are missing if the section above contains content.

Analyze the research notes and produce a clear analysis.

Look for:
1. Key findings
2. Agreements between sources
3. Disagreements between sources
4. Missing information or limitations
5. Unsupported or unreliable claims

Do not invent facts that are not contained in the research notes.

Only use NEEDS_MORE_RESEARCH if important information is genuinely
missing and additional web research could reasonably find it.

If more research is genuinely needed, your response MUST start with:

NEEDS_MORE_RESEARCH:

Then clearly explain exactly what information is missing.

Otherwise, provide the analysis directly.
"""

        if state["retry_count"] >= MAX_RETRIES:
            task += """

This is the final analysis attempt.
Do not request additional research.
Analyze the available research and clearly state any remaining limitations.
"""

        result = await run_agent(
            analyst,
            task,
            MAX_AGENT_STEPS,
        )

        answer = result["answer"]

        check_stage_output(answer)

        return {
            "analysis": answer,
            "stage_answers": state["stage_answers"] + [answer],
        }

    async def writer_node(state: PipelineState) -> dict:
        task = f"""
You are the final writer in a research pipeline.

Original question:
{state["query"]}

================ RESEARCH NOTES ================
{state["research"]}
================ END RESEARCH NOTES ================

================ ANALYSIS ================
{state["analysis"]}
================ END ANALYSIS ================

Write the final report using the research notes and analysis above.

Do NOT ask the user to provide research notes.
Do NOT ask the user to provide analysis.
Do NOT say that the research is missing when the sections above contain content.

Do not invent facts that are not supported by the provided material.

Write for someone who is new to the topic.

Requirements:
- Use clear headings.
- Explain difficult terms when they first appear.
- Present important findings clearly.
- Mention relevant disagreements or limitations.
- Keep the report concise but informative.
- End with a short conclusion explaining the main takeaway.
"""

        result = await run_agent(
            writer,
            task,
            MAX_AGENT_STEPS,
        )

        answer = result["answer"]

        check_stage_output(answer)

        return {
            "report": answer,
            "stage_answers": state["stage_answers"] + [answer],
        }

    def route_after_analyst(state: PipelineState) -> str:
        needs_more = state["analysis"].strip().startswith(
            "NEEDS_MORE_RESEARCH:"
        )

        if needs_more and state["retry_count"] < MAX_RETRIES:
            return "researcher"

        return "writer"

    def increment_retry(state: PipelineState) -> dict:
        return {
            "retry_count": state["retry_count"] + 1
        }

    graph = StateGraph(PipelineState)

    graph.add_node("researcher", researcher_node)
    graph.add_node("analyst", analyst_node)
    graph.add_node("writer", writer_node)
    graph.add_node("bump_retry", increment_retry)

    graph.add_edge(START, "researcher")
    graph.add_edge("researcher", "analyst")

    graph.add_conditional_edges(
        "analyst",
        route_after_analyst,
        {
            "researcher": "bump_retry",
            "writer": "writer",
        },
    )

    graph.add_edge("bump_retry", "researcher")
    graph.add_edge("writer", END)

    memory = InMemorySaver()
    pipeline = graph.compile(checkpointer=memory)

    initial_state = {
        "query": query,
        "research": "",
        "analysis": "",
        "report": "",
        "retry_count": 0,
        "stage_answers": [],
    }

    config = {
        "configurable": {
            "thread_id": "research_pipeline"
        },
        "recursion_limit": 20,
    }

    result = await pipeline.ainvoke(
        initial_state,
        config,
    )

    return {
        "answer": result["report"],
        "metadata": {
            "stages": 3,
            "retries": result["retry_count"],
            "stage_answers": len(result["stage_answers"]),
        },
    }

In [ ]:
# Live run — this calls the model and the web tools three times, so it takes a minute.
pipeline_result = await run_pipeline('Compare RAG and fine-tuning')
print(pipeline_result['answer'])
print(pipeline_result['metadata'])

In [ ]:
pipeline_result = await run_pipeline('What are the latest developments in Saudi Arabia\'s AI strategy under SDAIA and Vision 2030?')
print(pipeline_result['answer'])
print(pipeline_result['metadata'])

## 8. Check your run

Every run above printed a trace tree: one span per agent run, with the
token count and the real cost on it. This cell prints the pipeline metadata.

In [ ]:
if 'pipeline_result' not in globals():
    print('Finish TODO #2 and TODO #3 first, then run the pipeline cell.')
else:
    print('Pipeline metadata:', pipeline_result['metadata'])
    print('The trace trees above show the tokens and real cost for each stage.')

## 9. Challenges (these earn the Observability points)

Two optional upgrades. Uncomment the code, adapt it, and run it.

- **Stagnation detection**: `LoopDetector.check_output_stagnation` flags
  when stage outputs keep coming back nearly identical — a sign the
  pipeline is going in circles. `check_tool_call` flags repeated tool calls.
- **Memory**: pass a checkpointer to `create_agent` and a `thread_id` in
  the config, and the agent remembers earlier turns in that thread.

For Architecture points, redesign the pipeline graph itself — see EVALUATION.md.

In [ ]:
# # Challenge A — stagnation + repetition detection.
# # Run the three stages, keep each stage's answer, and compare them:
# # (collect each node result's answer in your pipeline, e.g. store stage answers in a dict as they run)
# #
# # detector = LoopDetector()
# # for stage_answer in [research_answer, analyst_answer, writer_answer]:
# #     check = detector.check_output_stagnation(stage_answer)
# #     if check.is_looping:
# #         print('Loop warning:', check.message)
# # Repetition: LoopDetector.check_tool_call('search_web', query)
# # returns is_looping=True when the same tool call repeats too often.
#
# # Challenge B — memory. An agent with a checkpointer remembers earlier turns.
# #
# # from langgraph.checkpoint.memory import InMemorySaver
# # memory_agent = create_agent(
# #     model=llm,
# #     tools=[],
# #     system_prompt='You are a helpful assistant.',
# #     checkpointer=InMemorySaver(),
# # )
# # config = {'configurable': {'thread_id': 'demo'}}
# # await memory_agent.ainvoke({'messages': [('user', 'My name is Sara.')]}, config=config)
# # reply = await memory_agent.ainvoke({'messages': [('user', 'What is my name?')]}, config=config)
# # print(reply['messages'][-1].content)